# 01 — Freeze the controlled DORIS-MAE supervision

This notebook reproduces the data operation in
`train_llama_seed.ipynb`: remove rows with no positive judgment,
sample three set-unique 50-document pools per eligible aspect row,
keep at most five positives, draw hard negatives from the first 100,
shuffle each pool deterministically, and truncate every paper to 300
words. It saves a hashed artifact with `query`, `docs`, and the
original graded `labels` (0/1/2).

Expected invariant: 1,501 input rows − 3 zero-positive rows = 1,498
eligible rows; 1,498 × 3 = **4,494 training examples**.
The original sampler's positive fallback makes 182 lists length 51;
the other 4,312 have the intended length 50. This is preserved so the
controlled baseline uses the actual LLM supervision.

In [ ]:
import os
import sys
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / "tools" / "modernbert_supervised_baseline.py").exists():
            return candidate
    raise FileNotFoundError(
        "Run this notebook from inside the cloned repository; "
        "tools/modernbert_supervised_baseline.py was not found."
    )

REPO_ROOT = find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f"Repository root: {REPO_ROOT}")

In [ ]:
import json
from pathlib import Path
import pandas as pd

from tools.modernbert_supervised_baseline import (
    materialize_listwise_supervision,
    read_jsonl,
    reproducibility_manifest,
    resolve_existing_path,
    sha256_file,
    write_json,
    write_jsonl,
)

DORIS_SUPERVISION_PATH = REPO_ROOT / "HRIA" / "reranking_dataset_doris_mae.jsonl"
DORIS_CORPUS_PATH = resolve_existing_path(
    env_name="DORIS_CORPUS_PATH",
    candidates=(
        Path("/DorisMAE/splits/Corpus.json"),
        Path("/DATA/j_andrei/DorisMAE/splits/Corpus.json"),
        REPO_ROOT / "DorisMAE" / "splits" / "Corpus.json",
        REPO_ROOT / "splits" / "Corpus.json",
    ),
)
print(f"Using DORIS corpus: {DORIS_CORPUS_PATH}")
OUTPUT_DIR = Path(os.environ.get(
    "MODERNBERT_ARTIFACT_DIR",
    str(REPO_ROOT / "artifacts" / "modernbert_supervised_baseline"),
))
SUPERVISION_OUTPUT = OUTPUT_DIR / "dorismae_listwise_4494.jsonl"
MANIFEST_OUTPUT = OUTPUT_DIR / "preparation_manifest.json"

EXPECTED_INPUT_ROWS = 1_501
EXPECTED_ELIGIBLE_ROWS = 1_498
EXPECTED_TRAINING_EXAMPLES = 4_494
EXPECTED_LIST_SIZE_HISTOGRAM = {"50": 4_312, "51": 182}
N_VARIANTS = 3
TARGET_SIZE = 50
MAX_POSITIVES = 5
HARD_NEG_TOP = 100
DOCUMENT_WORD_LIMIT = 300

for required_path in (DORIS_SUPERVISION_PATH, DORIS_CORPUS_PATH):
    if not required_path.exists():
        raise FileNotFoundError(f"Required input not found: {required_path}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

                The corpus reader below intentionally matches the original notebook:
                underscores in titles become spaces, and each document is formatted as
                `Title: ...
Abstract: ...`.

In [ ]:
supervision_rows = read_jsonl(DORIS_SUPERVISION_PATH)
assert len(supervision_rows) == EXPECTED_INPUT_ROWS, (
    f"Expected {EXPECTED_INPUT_ROWS} rows, found {len(supervision_rows)}. "
    "Stop: this is not the supervision artifact used by the LLM run."
)
assert all(
    len(row["candidate_pool"]) == len(row["candidate_scores"])
    for row in supervision_rows
)

print("Loading the DORIS-MAE corpus; this file is large.")
corpus_df = pd.read_json(DORIS_CORPUS_PATH)
required_columns = {"abstract_id", "title", "original_abstract"}
missing_columns = required_columns - set(corpus_df.columns)
if missing_columns:
    raise ValueError(f"Corpus is missing columns: {sorted(missing_columns)}")

abstract_lookup = {}
for row in corpus_df.itertuples(index=False):
    title = str(row.title).replace("_", " ")
    abstract_lookup[int(row.abstract_id)] = (
        f"Title: {title}\nAbstract: {row.original_abstract}"
    )
print(f"Loaded text for {len(abstract_lookup):,} papers.")

In [ ]:
examples, audit = materialize_listwise_supervision(
    supervision_rows,
    abstract_lookup,
    n_variants=N_VARIANTS,
    target_size=TARGET_SIZE,
    max_positives=MAX_POSITIVES,
    hard_neg_top=HARD_NEG_TOP,
    document_word_limit=DOCUMENT_WORD_LIMIT,
)

assert audit["eligible_rows"] == EXPECTED_ELIGIBLE_ROWS, audit
assert audit["removed_zero_positive_rows"] == 3, audit
assert len(examples) == EXPECTED_TRAINING_EXAMPLES, audit
assert audit["list_size_histogram"] == EXPECTED_LIST_SIZE_HISTOGRAM, audit
assert all(len(item["docs"]) == len(item["labels"]) for item in examples)
assert all(any(int(label) > 0 for label in item["labels"]) for item in examples)
assert {int(label) for item in examples for label in item["labels"]} <= {0, 1, 2}

print(json.dumps(audit, indent=2))
print("Example label histogram:", {
    label: sum(item["labels"].count(label) for item in examples)
    for label in (0, 1, 2)
})

In [ ]:
write_jsonl(SUPERVISION_OUTPUT, examples)
manifest = {
    "experiment": "controlled_supervised_compact_reranker",
    "source": "same DORIS-MAE-derived supervision as train_llama_seed.ipynb",
    "input_supervision_path": str(DORIS_SUPERVISION_PATH.resolve()),
    "input_supervision_sha256": sha256_file(DORIS_SUPERVISION_PATH),
    "input_corpus_path": str(DORIS_CORPUS_PATH.resolve()),
    "input_corpus_sha256": sha256_file(DORIS_CORPUS_PATH),
    "output_path": str(SUPERVISION_OUTPUT.resolve()),
    "output_sha256": sha256_file(SUPERVISION_OUTPUT),
    "pool_seed": "md5(source_row_id || query_text), first 8 hex digits",
    "audit": audit,
    "environment": reproducibility_manifest(),
}
write_json(MANIFEST_OUTPUT, manifest)
print(f"Saved supervision: {SUPERVISION_OUTPUT}")
print(f"Saved manifest:    {MANIFEST_OUTPUT}")
print(f"SHA-256:           {manifest['output_sha256']}")

Do not continue if an assertion fails. Copy the printed SHA-256 into
the experimental log and use this same file for every compact-baseline
run. LambdaLoss consumes the graded labels directly; it does not turn
equal labels into a deterministic total order.